In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit
from pathlib import Path

from dusty_colors.utils import load_jackknives, load_stack
import matplotlib.patches as mpatches

from astropy.cosmology import Planck18
import astropy.units as u

In [ ]:
def get_color(color):
    return {
        "g-i": "k",
        "g-r": "C0",
        "r-i": "C2",
        "i-z": "C3",
    }[color]


def arcmin_per_Mpc(z):
    D_A = Planck18.angular_diameter_distance(z).to(u.Mpc).value
    return (180 * 60) / (np.pi * D_A)


def add_angular_axis(ax, z=0.36):
    ax2 = ax.twiny()
    ax2.set_xscale("log")
    ax2.set_xlim(np.array(ax.get_xlim()) * arcmin_per_Mpc(z))
    ax2.set_xlabel(r"$\Delta\theta$ [arcmin]")
    return ax2

In [ ]:
results = load_jackknives("../results_main", "fcolors")

x = results[f"g-i_bin_centers"]
y_all = -2.5 * np.log10(results[f"g-i_avg"])
y_err_ = 2.5 / np.log(10) * results[f"g-i_err"] / results[f"g-i_avg"]

fig, ax = plt.subplots(figsize=(3, 3), dpi=200)

for i, (xi, yi, ei) in enumerate(zip(x, y_all, y_err_)):
    ls = {0: "-", 1: "--", 2: ":"}[i % 3]
    ax.errorbar(
        xi,
        yi,
        yerr=ei,
        marker=".",
        ls=ls,
        label=f"{i+1}",
        c=f"C{i // 3}",
    )

x_mean = x[0]
y_mean = y_all.mean(axis=0)
y_cov = np.cov(y_all, rowvar=False, bias=True) * (len(y_all) - 1)
y_err = np.sqrt(np.diag(y_cov))
ax.errorbar(
    x_mean,
    y_mean,
    yerr=y_err,
    marker="s",
    markerfacecolor="none",
    markersize=5,
    ls="",
    c="k",
    zorder=10,
)

ax.set(xscale="log", yscale="log")
ax.set(xlabel="$r_\perp$ [Mpc]", ylabel="$E(g-i)$ [mag]")
ax.legend(
    title="Jackknife region",
    ncols=3,
    handlelength=1.5,
    handletextpad=0.1,
    columnspacing=0.75,
    frameon=False,
)

add_angular_axis(ax)

ax.plot(full["g-i_bin_centers"], , c="r")

#fig.savefig("../figures/fig_result_jackknife.pdf", bbox_inches="tight")

In [ ]:
full = load_stack(
    Path("../results/stacks/full"),
    stack_type="fcolors",
    r_norm=None,
    correct_flip=True,
)
y_full = -2.5 * np.log10(full["g-i_avg"])

# Digitization of Menard Fig. 6
menard = np.array(
    [
        0.08101611782270254,
        0.16378937069540647,
        0.1445439770745927,
        0.08591439458053093,
        0.25451552912158953,
        0.035436477855785485,
        0.45409096109724745,
        0.03202539577230472,
        0.8101611782270257,
        0.01337745893148901,
        1.4265457829751846,
        0.008700825116137813,
        2.545155291215897,
        0.007017038286703837,
        4.540909610972479,
        0.004449914313113396,
        8.101611782270266,
        0.003326427190621989,
        14.26545782975184,
        0.002275845926074791,
        25.45155291215894,
        0.0009874281281804155,
        0.08101611782270263,
        0.19306977288832497,
        0.14454397707459266,
        0.10386841787287614,
        0.2545155291215898,
        0.044499143131133935,
        0.45409096109724756,
        0.0372759372031494,
        0.810161178227026,
        0.01701254279852591,
        1.4454397707459266,
        0.011205998227840196,
        2.545155291215897,
        0.008591439458053098,
        4.540909610972477,
        0.005804118407660814,
        7.995712002123401,
        0.004177140862395749,
        14.265457829751824,
        0.0030062284327739,
        25.45155291215894,
        0.0015570684047537333,
    ]
)
menard = menard.reshape(-1, 2)

menard_x = menard[:11, 0] / arcmin_per_Mpc(3.6)
menard_y = menard[:11, 1]
menard_err = menard[11:, 1] - menard_y

# Above was digitized from incorrect y-axis. Need to divide by 10
menard_y /= 10
menard_err /= 10

# Genc Av from Fig. 11
kids = np.array(
    [
        0.2923623220416735,
        0.011429109934265547,
        1.3901168733868028,
        0.002768473614393809,
        2.9919742049233986,
        0.0017037253536931446,
        6.54985454658025,
        0.0009731964854250654,
        14.212295777240813,
        0.0007037756813106066,
        0.2895907556713154,
        0.020997654572143913,
        1.3772822449804694,
        0.00443710166397699,
        2.989125669537237,
        0.0028695607540067582,
        6.54257996945615,
        0.001787937537768105,
        14.197476846883712,
        0.0012457001211372406,
    ]
)
kids = kids.reshape(10, 2)

kids_x = kids[:5, 0] / arcmin_per_Mpc(0.3)
kids_y = kids[:5, 1]
kids_err = kids[5:, 1] - kids_y

kids_y *= (3.66 - 2.05) / 3.1
kids_err *= (3.66 - 2.05) / 3.1

# Genc g-i from Fig C.2
kids_gi = np.array(
    [
        0.2658633507876159,
        0.008972980775916186,
        0.5821505188557653,
        0.007214561089854725,
        1.2678339316650937,
        0.002248549806308625,
        2.7616917937837737,
        0.000613230287853244,
        0.2656691794262092,
        0.014703859029447723,
        0.5819206589901929,
        0.009422213449339233,
        1.2669830304822378,
        0.00354001947591735,
        2.7588030603933844,
        0.0012441496120107712,
    ]
)
kids_gi = kids_gi.reshape(8, 2)

kids_x_gi = kids_gi[:4, 0]
kids_y_gi = kids_gi[:4, 1]

kids_err_gi = kids_gi[4:, 1] - kids_y_gi

# Convert arcmin -> Mpc
kids_x_gi /= arcmin_per_Mpc(0.3)

fig, ax1 = plt.subplots(figsize=(3, 3), dpi=200)

x = np.logspace(-2, 1, 100)
y = 1.4e-3 * (x * arcmin_per_Mpc(0.36)) ** (-0.84)
ax1.plot(x, y, c="k", ls="--", lw=0.75, zorder=-1)


ax1.set(
    xscale="log",
    yscale="log",
    xlabel="$r_\perp$ (Mpc)",
    ylabel="$E(g-i)$ [mag]",
    # ylim=(4e-4, 0.3),
    # xlim=(x.min(), x.max()),
)
ax1.set_box_aspect(1)

ax1.errorbar(
    menard_x,
    menard_y,
    yerr=menard_err,
    c="dimgray",
    mfc="w",
    ls="",
    marker=".",
    label="Menard 2010",
)

"""ax1.errorbar(
    kids_x,
    kids_y,
    yerr=kids_err,
    c="silver",
    mfc="w",
    ls="",
    marker=".",
    label="Genc 2025",
)"""
ax1.errorbar(
    kids_x_gi,
    kids_y_gi,
    yerr=kids_err_gi,
    c="silver",
    mfc="w",
    ls="",
    marker=".",
    label="Genc 2025",
)

ax1.errorbar(
    x_mean,
    y_full,
    yerr=y_err,
    color="r",
    label="Rubin DP1",
    ls="",
    marker="s",
    markersize=3,
)

ax1.legend(loc="upper right", fontsize=7, frameon=False)


def power_law(x, A, alpha):
    return A * x**alpha


popt, pcov = curve_fit(
    power_law, x_mean, y_full, p0=[0.003, -0.86], sigma=y_cov, absolute_sigma=True
)
print("Best fit power law:")
print(f"    scale = {popt[0]:.1e} ± {np.sqrt(pcov[0, 0]):.1e}")
print(f"    index = {popt[1]:.2f} ± {np.sqrt(pcov[1, 1]):.2f}")
x_ = np.logspace(-2, 1, 100)
ax1.plot(x_, power_law(x_, *popt), c="r", ls="--", lw=0.75, zorder=-1)

add_angular_axis(ax1)

# fig.savefig("../figures/fig_result_compare.pdf", bbox_inches="tight")

In [ ]:
4.7e-03 / (1.4e-3 * arcmin_per_Mpc(0.36) ** (-0.84))

In [ ]:
results = load_jackknives("../results_main", "fcolors")

fig, ax = plt.subplots(figsize=(3, 3), dpi=200)

for i, color in enumerate(["g-i", "g-r", "r-i", "i-z"]):
    x = results[f"{color}_bin_centers"][0]

    y_all = -2.5 * np.log10(results[f"{color}_avg"])
    y = y_all.mean(axis=0)
    y_cov = np.cov(y_all.T, bias=True) * (len(y_all) - 1)
    y_err = np.sqrt(np.diag(y_cov))

    offset = 1 + ((i % 2) - 0.5) * 0.08
    ax.errorbar(
        x * offset,
        y,
        yerr=y_err,
        marker=".",
        ls="",
        label=f"${color}$",
        color=get_color(color),
    )

ax.set(xscale="log", yscale="log", ylim=(4e-4, 0.3))
ax.set(xlabel="$r_\perp$ [Mpc]", ylabel="$E(X-Y\,)$ [mag]")

# Create custom two-column legend
handles, labels = ax.get_legend_handles_labels()
blank = mpatches.Patch(color="none", label="")
new_handles = [handles[0], blank, blank, handles[1], handles[2], handles[3]]
new_labels = [labels[0], "", "", labels[1], labels[2], labels[3]]
ax.legend(
    new_handles,
    new_labels,
    ncol=2,
    frameon=False,
    handletextpad=0.1,
    columnspacing=0.5,
)

add_angular_axis(ax)

fig.savefig("../figures/fig_result_colors.pdf", bbox_inches="tight")

In [ ]:
fig, ax = plt.subplots(figsize=(3, 3), dpi=200)

for i, subset in enumerate(["red", "blue"]):
    results = load_jackknives(f"../results_{subset}", "fcolors")

    x = results["g-i_bin_centers"][0]
    y_all = -2.5 * np.log10(results["g-i_avg"])
    y = y_all.mean(axis=0)
    y_cov = np.cov(y_all.T, bias=True) * (len(y_all) - 1)
    y_err = np.sqrt(np.diag(y_cov))

    # Plot upper limit for missing point
    if subset == "blue":
        ax.errorbar(
            x[1],
            y[1] + 3 * y_err[1],
            yerr=y_err[1],
            uplims=True,
            marker="_",
            color="b",
        )
        y[1] = np.nan

    offset = 1 + ((i % 2) - 0.5) * 0.08
    ax.errorbar(
        x * offset,
        y,
        yerr=y_err,
        marker=".",
        ls="",
        label=subset,
        color=subset,
    )

ax.set(xscale="log", yscale="log", ylim=(4e-4, 0.3))
ax.set(xlabel="$r_\perp$ [Mpc]", ylabel="$E(g-i)$ [mag]")
ax.legend(frameon=False, handletextpad=0.1)

# x_ = np.logspace(-2, 0, 100)
# ax.plot(x_, power_law(x_, *popt), c="k", ls="--", lw=0.75, zorder=-1)

add_angular_axis(ax)

fig.savefig("../figures/fig_result_red_vs_blue.pdf", bbox_inches="tight")